# Task 2 — Notebook 01: CDL Multi-Year Time Series Loading

**Purpose (NAFSI Task 2):** Identify **regular crop rotation** (e.g. corn–soy alternation) from a **10-year CDL series** in the Corn Belt. This notebook loads the **full interim CDL stack** and slices the configured year window.

**Inputs (interim):**  
- `data/interim/cdl/cdl_stack_{y0}_{y1}.nc` — dims `year`, `y`, `x`

**Outputs:** in-memory `cdl_stack` for rotation metrics (subset years from `configs/task2_crop_rotation.yaml`). State masking can be added using `data/external/states/` in a later cell or notebook.

In [ ]:
# NOTE: This notebook was scaffolded with AI assistance.

import sys
from pathlib import Path

import yaml

_cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / "requirements.txt").is_file() and (p / "src").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not find repo root. cd to the repo and retry.")
sys.path.insert(0, str(REPO_ROOT))

from src.io.interim_loaders import load_cdl_stack_from_interim

with open(REPO_ROOT / "configs" / "task2_crop_rotation.yaml") as f:
    cfg = yaml.safe_load(f)

y0, y1 = int(cfg["cdl"]["year_range"][0]), int(cfg["cdl"]["year_range"][1])
print("Configured CDL window:", y0, "–", y1)

In [ ]:
cdl_full = load_cdl_stack_from_interim(REPO_ROOT)
years_avail = [int(y) for y in cdl_full["year"].values]
cdl_stack = cdl_full.sel(year=slice(y0, y1))
print("Full stack years:", min(years_avail), "…", max(years_avail), "count", len(years_avail))
print("Task 2 subset", dict(cdl_stack.sizes), cdl_stack.dtype)


In [ ]:
corn, soy = cfg["cdl"]["crop_classes"]["corn"], cfg["cdl"]["crop_classes"]["soybean"]
for y in cdl_stack["year"].values.astype(int):
    frac_corn = float((cdl_stack.sel(year=y) == corn).mean())
    frac_soy = float((cdl_stack.sel(year=y) == soy).mean())
    print(f"year {y}: P(corn)={frac_corn:.4f} P(soy)={frac_soy:.4f}")
